In [2]:
# Cell 0: synchronize the repository environment before running the workbench.
import subprocess
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Find the workspace root without confusing member pyproject files for it."""
    for candidate in (start, *start.parents):
        if (
            (candidate / "uv.lock").exists()
            and (candidate / "implementations").is_dir()
            and (candidate / "aieng-forecasting").is_dir()
        ):
            return candidate
    raise FileNotFoundError("Could not find the agentic-forecasting repository root.")


SETUP_ROOT = find_repo_root(Path.cwd().resolve())
print("Running: uv sync --all-extras --dev --all-packages")
setup_result = subprocess.run(
    ["uv", "sync", "--all-extras", "--dev", "--all-packages"],
    cwd=SETUP_ROOT,
    check=False,
    text=True,
    capture_output=True,
)
print(setup_result.stdout)
if setup_result.returncode != 0:
    print(setup_result.stderr)
    raise RuntimeError("uv sync failed. Install uv or restart the notebook with the repository environment selected.")
print("Environment synchronized successfully.")
print("If packages changed, restart the notebook kernel before continuing.")

Running: uv sync --all-extras --dev --all-packages

Environment synchronized successfully.
If packages changed, restart the notebook kernel before continuing.


# Manufacturing Stress Parameter Sweep Workbench

This copied workbench evaluates the deterministic statistical candidates without LLM calls or prediction-result caches.

The tuning stage compares historical frequency, six logistic regularization settings, and four small XGBoost settings across three chronological folds within 2000–2017. Each forecast is still fit using only data available at its own origin.

The fixed 2018–2024 confirmation window was examined by an earlier run. Treat it as a historical diagnostic, not as an untouched holdout or a basis for selecting new candidates. A clean next confirmation needs future or prospectively recorded outcomes.

In [ ]:
# Main controls: change these values, then run the notebook from top to bottom.
STAGE = "confirm"
CANDIDATE = "logistic_c_0_001"
CANDIDATE = "xgb_50_depth2_lr0_03"
BACKTEST_STRIDE = 3
REFRESH_INPUT_DATA = False
RUN_SWEEP = True
SHOW_ORIGIN_DETAILS = False

# For confirmation, use only a candidate that passes the tuning stability rule.
# Keep BACKTEST_STRIDE unchanged between stages.
print(
    {
        "stage": STAGE,
        "candidate": CANDIDATE,
        "backtest_stride": BACKTEST_STRIDE,
        "refresh_input_data": REFRESH_INPUT_DATA,
        "run_sweep": RUN_SWEEP,
        "show_origin_details": SHOW_ORIGIN_DETAILS,
    }
)

{'stage': 'confirm', 'candidate': 'logistic_c_0_001', 'backtest_stride': 3, 'refresh_input_data': False, 'run_sweep': True, 'show_origin_details': False}


## Imports and active features

The notebook imports the candidate definitions and evaluation helpers from `run_parameter_smoke.py`; it does not duplicate the sweep logic. The active feature list is printed so changes in `features.py` are visible before the run starts.


In [22]:
from importlib import reload

import manufacturing_stress_forecasting.run_parameter_smoke as parameter_smoke
import yaml
from aieng.forecasting.evaluation import BacktestSpec
from dotenv import load_dotenv
from IPython.display import display as ipython_display
from manufacturing_stress_forecasting.data import build_manufacturing_stress_service
from manufacturing_stress_forecasting.features import FEATURE_SERIES_IDS


parameter_smoke = reload(parameter_smoke)
REPO_ROOT = SETUP_ROOT
load_dotenv(REPO_ROOT / ".env", override=False)
SPEC_PATH = (
    REPO_ROOT / "implementations" / "manufacturing_stress_forecasting" / "specs" / "manufacturing_stress_smoke.yaml"
)
print(f"Repository: {REPO_ROOT}")
print(f"Active statistical features ({len(FEATURE_SERIES_IDS)}):")
for feature_name in FEATURE_SERIES_IDS:
    print(f"  - {feature_name}")

Repository: /home/coder/BMO-C2-agentic-forecasting-1
Active statistical features (15):
  - ipman_change_1m_pct
  - ipman_change_3m_pct
  - ipman_change_6m_pct
  - FEDFUNDS
  - YC_SPREAD
  - CPIAUCSL
  - CPI_YOY
  - UNRATE
  - ICSA
  - VIXCLS
  - CREDIT_SPREAD
  - SPY_RETURN_3M
  - SPY_RETURN_12M
  - XLI_RETURN_3M
  - XLI_RETURN_12M


## Create or refresh the data service

The default reads the existing local FRED and Yahoo Finance caches. Set `REFRESH_INPUT_DATA = True` only when you intentionally want to download fresh source data.


In [23]:
service = build_manufacturing_stress_service(
    cache_dir=REPO_ROOT / "data" / "fred",
    yahoo_cache_dir=REPO_ROOT / "data" / "yfinance",
    refresh=REFRESH_INPUT_DATA,
)
ipython_display(service.summary())

,series_id,description,source,units,frequency,n_obs,start,end
0,CPIAUCSL,Consumer Price Index for All Urban Consumers,FRED (CPIAUCSL),Index,MS,955,1947-01-01,2026-08-01
1,CPI_YOY,Consumer Price Index year-over-year change,Derived from FRED (CPIAUCSL),Percent,MS,943,1948-01-01,2026-08-01
2,CREDIT_SPREAD,Moody's Baa corporate bond yield minus 10-year...,FRED (BAA10Y),Percentage points,MS,489,1986-01-01,2026-09-01
3,FEDFUNDS,Effective federal funds rate,FRED (FEDFUNDS),Percent,MS,867,1954-07-01,2026-09-01
4,ICSA,Initial claims for unemployment insurance,FRED (ICSA),Number,MS,717,1967-01-01,2026-09-01
5,SPY_RETURN_12M,SPY trailing 12-month adjusted-close return,"Yahoo Finance (SPY), derived",Percent,MS,334,1999-01-01,2026-10-01
6,SPY_RETURN_3M,SPY trailing 3-month adjusted-close return,"Yahoo Finance (SPY), derived",Percent,MS,343,1998-04-01,2026-10-01
7,UNRATE,U.S. unemployment rate,FRED (UNRATE),Percent,MS,943,1948-01-01,2026-08-01
8,VIXCLS,CBOE volatility index,FRED (VIXCLS),Index,MS,441,1990-01-01,2026-09-01
9,XLI_RETURN_12M,XLI trailing 12-month adjusted-close return,"Yahoo Finance (XLI), derived",Percent,MS,323,1999-12-01,2026-10-01


## Build the tuning or confirmation experiment

The stage determines the date window and candidate set. Tuning runs the historical-frequency baseline plus every configured candidate. Confirmation runs the baseline plus exactly one selected candidate.


In [24]:
if STAGE not in {"tune", "confirm"}:
    raise ValueError("STAGE must be 'tune' or 'confirm'.")
if BACKTEST_STRIDE not in {1, 3}:
    raise ValueError("BACKTEST_STRIDE must be 1 or 3.")
if STAGE == "confirm" and CANDIDATE is None:
    raise ValueError("Set CANDIDATE to the tuning winner before confirmation.")
if STAGE == "tune" and CANDIDATE is not None:
    raise ValueError("Leave CANDIDATE as None during tuning.")

with SPEC_PATH.open() as file:
    base_spec = BacktestSpec.model_validate(yaml.safe_load(file))

spec = parameter_smoke.build_experiment_spec(base_spec, stage=STAGE, stride=BACKTEST_STRIDE)
named_predictors = parameter_smoke.predictors_for_stage(STAGE, CANDIDATE)

print(
    f"Stage={STAGE}; origins={spec.start.date()} to {spec.end.date()}; "
    f"stride={spec.stride}; horizon={spec.task.horizons[0]} month(s)"
)
print("Predictors:")
for predictor_name, _ in named_predictors:
    print(f"  - {predictor_name}")

if STAGE == "tune":
    print("Configured candidates:")
    for candidate in parameter_smoke.CANDIDATES:
        print(f"  - {candidate.name}")

Stage=confirm; origins=2018-01-01 to 2024-12-01; stride=3; horizon=3 month(s)
Predictors:
  - historical_frequency
  - logistic_c_0_001


## Run the parameter sweep

Tuning runs 11 predictors across three folds, so it can take a while. Set `RUN_SWEEP = False` to inspect the setup without running models. Set `SHOW_ORIGIN_DETAILS = True` only when you need every forecast probability and resolved label.

In [25]:
results = None
fold_results = None
fold_table = None
pooled_table = None
origin_details = None
table = None
if RUN_SWEEP:
    if STAGE == "tune":
        fold_results = parameter_smoke.run_tuning_folds(
            base_spec=base_spec,
            stride=BACKTEST_STRIDE,
            service=service,
        )
        fold_table, pooled_table, origin_details = parameter_smoke.tuning_summary_tables(
            fold_results,
            service=service,
        )
    else:
        results = parameter_smoke.run_candidates(named_predictors, spec=spec, service=service)
        table = parameter_smoke.comparison_table(results)
else:
    print("Sweep skipped because RUN_SWEEP is False.")

Running historical_frequency...
Running logistic_c_0_001...


## Compare Brier scores

The first table reports Brier score, skill against historical frequency, and stress-event counts by fold. The second pools the tuning folds and reports how many folds, including event-bearing folds, each candidate beats the baseline in. Lower Brier is better.

A candidate is suggested only when it improves pooled Brier, beats the baseline in at least two folds, and improves on an event-bearing fold. If stress events appear in fewer than two folds, treat any selected candidate as provisional. `SHOW_ORIGIN_DETAILS` reveals the forecast-level probabilities, labels, and Brier scores.

In [26]:
if STAGE == "tune" and fold_results is not None:
    fold_table, pooled_table, origin_details = parameter_smoke.tuning_summary_tables(fold_results, service=service)
    ipython_display(
        fold_table.style.format(
            {
                "mean_brier": "{:.4f}",
                "delta_vs_baseline": "{:+.4f}",
                "brier_skill": "{:+.1%}",
            }
        )
    )
    ipython_display(
        pooled_table.style.format(
            {
                "mean_brier": "{:.4f}",
                "delta_vs_baseline": "{:+.4f}",
                "brier_skill": "{:+.1%}",
            }
        )
    )

    selected_model = parameter_smoke.select_tuning_candidate(pooled_table)
    if selected_model is None:
        print(
            "No candidate beat historical frequency overall, in at least two folds, "
            "and in an event-bearing fold. Keep historical frequency."
        )
    else:
        selected_candidate = str(selected_model["candidate"])
        print(f"Tuning candidate passing the stability rule: {selected_candidate}")
        if int(selected_model["event_folds"]) < 2:
            print(
                "Caution: stress events appear in fewer than two folds; treat this candidate as provisional."
            )
        print("To reproduce the historical diagnostic, set:")
        print('STAGE = "confirm"')
        print(f'CANDIDATE = "{selected_candidate}"')
        print(f"BACKTEST_STRIDE = {BACKTEST_STRIDE}")
        print("The 2018–2024 window has already been inspected and is not an untouched confirmation set.")

    if SHOW_ORIGIN_DETAILS:
        ipython_display(origin_details)
elif STAGE == "confirm" and table is not None:
    ipython_display(
        table.style.format(
            {
                "mean_brier": "{:.4f}",
                "delta_vs_baseline": "{:+.4f}",
                "brier_skill": "{:+.1%}",
            }
        )
    )
    print(f"Historical diagnostic completed for {CANDIDATE}; do not tune against this window.")

,candidate,mean_brier,delta_vs_baseline,brier_skill,scored,skipped
0,historical_frequency,0.0356,+0.0000,+0.0%,28,0
1,logistic_c_0_001,0.0380,+0.0024,-6.7%,28,0


Historical diagnostic completed for logistic_c_0_001; do not tune against this window.


## Confirmation discipline

The existing `confirm` stage evaluates one candidate on 2018–2024, but that period has already been inspected. Use it only to reproduce or diagnose the historical result; do not choose parameters from it. For a fresh confirmation, freeze the candidate first and evaluate it only on future or prospectively recorded outcomes.